# 10 — P2: Train Lipschitz-Margin ViT-Tiny on MNIST

**Plan 2 — Phase 1.3**: Train the *same* ViT-Tiny architecture as notebook 09, but with:
1. **Spectral normalization** on every `Linear` and `Conv2d` layer (bounds each layer's Lipschitz constant ≤ 1).
2. **Margin loss** (multi-class hinge) added to cross-entropy — pushes logit margins wide so the global Lipschitz pre-filter (Phase 6) certifies more samples.

Reference: LipShiFT (https://arxiv.org/abs/2503.14751, https://github.com/RohanMenon/LipShiFT). We do **not** replace MHSA with shift modules — the goal of Plan 2 is to verify the standard MHSA encoder block.

## Output
Checkpoint at `runs/vit_tiny_lipmargin/model.pt`. Target ≥92% test accuracy; small global Lipschitz bound (measured in Phase 2).

In [2]:
!pip install -q numpy pandas torch torchvision tqdm pyyaml

In [3]:
# ── Imports & seed ────────────────────────────────────────────────────────
from __future__ import annotations
import math, json, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed: int = 1234) -> None:
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(1234)
print(f'Device : {device}\nPyTorch: {torch.__version__}')


Device : cuda
PyTorch: 2.10.0+cu128


In [4]:
# ── Config ────────────────────────────────────────────────────────────────
CFG = dict(
    seed         = 1234,
    data_root    = '/tmp/mnist',
    run_dir      = r'C:\Users\manya\OneDrive\Desktop\THESIS\formal-verification\runs\vit_tiny_lipmargin',
    n_epochs     = 30,
    batch_size   = 128,
    lr           = 3e-4,
    weight_decay = 0.01,
    # Architecture (must match notebook 09 exactly)
    img_size     = 28,
    patch_size   = 4,
    embed_dim    = 64,
    num_heads    = 2,
    num_layers   = 2,
    mlp_ratio    = 2,
    eps_rms      = 1e-6,
    # Lipschitz-margin training
    margin_kappa = 2.0,    # hinge margin κ — wider = stronger Lipschitz incentive
    lambda_marg  = 1.0,    # weight on margin term (CE has weight 1.0)
    lambda_sigma = 0.05,   # NEW: penalty on σ_max so σ is pushed DOWN, not just capped at 1
    sn_n_iter    = 3,      # power-iteration steps for soft spectral cap (more = tighter σ)
)
Path(CFG['run_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2))


{
  "seed": 1234,
  "data_root": "/tmp/mnist",
  "run_dir": "C:\\Users\\manya\\OneDrive\\Desktop\\THESIS\\formal-verification\\runs\\vit_tiny_lipmargin",
  "n_epochs": 30,
  "batch_size": 128,
  "lr": 0.0003,
  "weight_decay": 0.01,
  "img_size": 28,
  "patch_size": 4,
  "embed_dim": 64,
  "num_heads": 2,
  "num_layers": 2,
  "mlp_ratio": 2,
  "eps_rms": 1e-06,
  "margin_kappa": 2.0,
  "lambda_marg": 1.0,
  "lambda_sigma": 0.05,
  "sn_n_iter": 3
}


In [5]:
# ── Data ──────────────────────────────────────────────────────────────────
def get_mnist_loaders(batch_size=128, data_root='/tmp/mnist'):
    tf = transforms.ToTensor()
    train_ds = torchvision.datasets.MNIST(data_root, train=True,  download=True, transform=tf)
    test_ds  = torchvision.datasets.MNIST(data_root, train=False, download=True, transform=tf)
    kw = dict(num_workers=0, pin_memory=False)
    return (
        torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw),
        torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw),
    )

train_loader, test_loader = get_mnist_loaders(CFG['batch_size'], CFG['data_root'])
print(f'Train: {len(train_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 468kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.91MB/s]

Train: 60,000  Test: 10,000


In [6]:
# ── ViT-Tiny model (identical architecture to notebook 09) ────────────────
#
# Spectral normalization is applied AFTER construction (cell below) so the
# class definition stays one-to-one with notebook 09. Phase 2 / 4 / 5 phases
# can load either checkpoint into the same class.

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class MHSA(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale    = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn   = F.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)


class MLPBlock(nn.Module):
    def __init__(self, embed_dim: int, mlp_ratio: int = 2):
        super().__init__()
        hidden = embed_dim * mlp_ratio
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int = 2,
                 eps_rms: float = 1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms)
        self.attn  = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms)
        self.mlp   = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        x = self.norm(x).mean(dim=1)
        return self.head(x)

print('Model class defined')

Model class defined


In [7]:
# ── Soft spectral norm cap (σ ≤ 1, NOT σ ≡ 1) ─────────────────────────────
#
# CRITICAL FIX: torch.nn.utils.spectral_norm divides W by σ unconditionally,
# which forces σ exactly to 1.  That is WRONG for our purpose: we want a
# Lipschitz UPPER BOUND, so we want σ ≤ 1.  If a layer is naturally below 1
# (e.g. W_V at σ ≈ 0.45 in the standard run), forcing σ = 1 actually MAKES
# the global Lipschitz bound LARGER, not smaller — exactly what was breaking
# the previous lipmargin run (σ_V went 0.45 → 1.0, σ_O went 0.49 → 1.0).
#
# Soft cap:    W_eff  =  W / max(1, σ(W))
#   σ(W) ≤ 1  →  divisor = 1   →  weight unchanged
#   σ(W) > 1  →  divisor = σ   →  σ(W_eff) = 1
#
# Implemented as a `torch.nn.utils.parametrize` parametrization so the
# raw weight (`weight_orig`) is what the optimizer updates, and the
# materialized weight (used in forward / saved separately) has σ ≤ 1.

from torch.nn.utils import parametrize

class SoftSpectralCap(nn.Module):
    """Forward: W → W / max(1, σ(W)) via power iteration."""
    def __init__(self, weight: torch.Tensor, n_iter: int = 1):
        super().__init__()
        self.n_iter = n_iter
        with torch.no_grad():
            W2d = weight.detach().reshape(weight.shape[0], -1)
            u = torch.randn(W2d.shape[0], device=weight.device)
            u = u / (u.norm() + 1e-12)
        self.register_buffer('u', u)

    def forward(self, weight: torch.Tensor) -> torch.Tensor:
        W2d = weight.reshape(weight.shape[0], -1)
        u = self.u
        # Power iteration without grad to update u
        with torch.no_grad():
            for _ in range(self.n_iter):
                v = W2d.t() @ u; v = v / (v.norm() + 1e-12)
                u = W2d @ v;     u = u / (u.norm() + 1e-12)
            self.u.copy_(u)
        # Differentiable σ estimate (gradient flows through W2d only)
        v = W2d.t() @ u; v = v / (v.norm() + 1e-12)
        sigma = u @ (W2d @ v)
        # Soft cap: only divide if σ > 1
        scale = torch.clamp(sigma, min=1.0)
        return weight / scale

    def right_inverse(self, weight: torch.Tensor) -> torch.Tensor:
        return weight


def apply_soft_spectral_cap(model: nn.Module, n_iter: int = 1) -> int:
    n = 0
    for parent in model.modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, (nn.Linear, nn.Conv2d)):
                parametrize.register_parametrization(
                    child, 'weight', SoftSpectralCap(child.weight, n_iter)
                )
                n += 1
    return n


set_seed(CFG['seed'])
model = ViTTiny(
    img_size=CFG['img_size'], patch_size=CFG['patch_size'],
    embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'],
    num_layers=CFG['num_layers'], mlp_ratio=CFG['mlp_ratio'],
    eps_rms=CFG['eps_rms'],
).to(device)
n_wrapped = apply_soft_spectral_cap(model, CFG['sn_n_iter'])
n_params  = sum(p.numel() for p in model.parameters())
print(f'Soft-spectral-cap wrapped {n_wrapped} layers (σ ≤ 1, naturally small σ preserved)')
print(f'Total parameters         : {n_params:,}')

# Smoke check
with torch.no_grad():
    _x = torch.zeros(2, 1, 28, 28, device=device)
    _y = model(_x)
    print(f'Output shape={tuple(_y.shape)}  finite={torch.isfinite(_y).all().item()}')

# σ summary at init (every layer should already be ≤ 1.0 after the cap)
@torch.no_grad()
def sigma_summary(m, tag):
    print(f'\nσ_max ({tag}):')
    for nm, p in m.named_modules():
        if isinstance(p, (nn.Linear, nn.Conv2d)):
            W = p.weight.detach().reshape(p.weight.shape[0], -1)
            s = float(torch.linalg.svdvals(W).max())
            flag = '' if s <= 1.001 else '   ← > 1 (cap not engaged?)'
            print(f'  {nm:40s} σ={s:.3f}{flag}')

sigma_summary(model, 'after init, after cap')


Soft-spectral-cap wrapped 14 layers (σ ≤ 1, naturally small σ preserved)
Total parameters         : 71,370
Output shape=(2, 10)  finite=True

σ_max (after init, after cap):
  patch_embed.proj                         σ=0.236
  blocks.0.attn.W_q                        σ=0.314
  blocks.0.attn.W_k                        σ=0.303
  blocks.0.attn.W_v                        σ=0.312
  blocks.0.attn.W_o                        σ=0.311
  blocks.0.mlp.fc1                         σ=0.373
  blocks.0.mlp.fc2                         σ=0.376
  blocks.1.attn.W_q                        σ=0.306
  blocks.1.attn.W_k                        σ=0.299
  blocks.1.attn.W_v                        σ=0.305
  blocks.1.attn.W_o                        σ=0.315
  blocks.1.mlp.fc1                         σ=0.372
  blocks.1.mlp.fc2                         σ=0.373
  head                                     σ=0.214


In [8]:
# ── Loss: CE + multi-class hinge margin + σ-penalty ───────────────────────
#
# margin_loss = mean_i  max(0,  max_{c≠y_i} z_c  -  z_{y_i}  +  κ )
#
# Wider margins (high κ, large λ_marg) → model logits are less sensitive to
# input perturbations of size ε for any given Lipschitz constant L,
# so more samples pass the Phase 6 Lipschitz pre-filter
# (margin > L · ε · √D).
#
# σ-penalty = mean_layer  σ_max(W)
# Adds a CONTINUOUS DOWNWARD pressure on σ even after it drops below the
# soft cap of 1.0 — without this, σ_V / σ_O would just stay at their random
# init value (~0.5) instead of decreasing further during training.

def margin_loss(logits: torch.Tensor, y: torch.Tensor, kappa: float) -> torch.Tensor:
    z_y      = logits.gather(1, y.unsqueeze(1)).squeeze(1)
    masked   = logits.clone()
    masked.scatter_(1, y.unsqueeze(1), float('-inf'))
    z_other  = masked.max(dim=1).values
    return F.relu(z_other - z_y + kappa).mean()


def sigma_penalty(model: nn.Module) -> torch.Tensor:
    """
    Mean of σ_max over Linear / Conv2d layers, computed with ONE step of
    power iteration on the materialized weight (post-cap).  Differentiable.
    """
    sigmas = []
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            W = m.weight                                      # post-cap weight
            W2d = W.reshape(W.shape[0], -1)
            with torch.no_grad():
                u = torch.randn(W2d.shape[0], device=W.device)
                u = u / (u.norm() + 1e-12)
                for _ in range(2):
                    v = W2d.t() @ u; v = v / (v.norm() + 1e-12)
                    u = W2d @ v;     u = u / (u.norm() + 1e-12)
            v = W2d.t() @ u; v = v / (v.norm() + 1e-12)
            sigmas.append(u @ (W2d @ v))
    return torch.stack(sigmas).mean()


def total_loss(model: nn.Module, logits: torch.Tensor, y: torch.Tensor,
               kappa: float, lam_marg: float, lam_sigma: float):
    ce  = F.cross_entropy(logits, y)
    mar = margin_loss(logits, y, kappa)
    sig = sigma_penalty(model)
    loss = ce + lam_marg * mar + lam_sigma * sig
    return loss, ce.detach(), mar.detach(), sig.detach()


In [9]:
# ── Train loop ────────────────────────────────────────────────────────────
@torch.no_grad()
def eval_acc_and_margin(model, loader, device):
    model.eval()
    correct = total = 0
    margins = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        z = model(x)
        correct += (z.argmax(1) == y).sum().item()
        total   += len(y)
        z_y     = z.gather(1, y.unsqueeze(1)).squeeze(1)
        m       = z.clone(); m.scatter_(1, y.unsqueeze(1), float('-inf'))
        margins.append((z_y - m.max(dim=1).values).cpu())
    margins = torch.cat(margins)
    return correct / total, margins.mean().item(), margins.median().item()


opt   = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['n_epochs'])

best_acc, best_state = 0.0, None
history = []
for ep in range(1, CFG['n_epochs'] + 1):
    t0 = time.time()
    model.train()
    sum_loss = sum_ce = sum_mar = sum_sig = n = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        z = model(x)
        loss, ce, mar, sig = total_loss(model, z, y,
                                         CFG['margin_kappa'],
                                         CFG['lambda_marg'],
                                         CFG['lambda_sigma'])
        loss.backward()
        opt.step()
        bs = len(y)
        sum_loss += loss.item() * bs
        sum_ce   += ce.item()   * bs
        sum_mar  += mar.item()  * bs
        sum_sig  += sig.item()  * bs
        n        += bs
    sched.step()

    test_acc, mean_margin, median_margin = eval_acc_and_margin(model, test_loader, device)
    history.append(dict(epoch=ep,
                        train_loss=sum_loss/n, train_ce=sum_ce/n,
                        train_margin=sum_mar/n, train_sigma=sum_sig/n,
                        test_acc=test_acc, test_margin_mean=mean_margin,
                        test_margin_median=median_margin,
                        lr=opt.param_groups[0]['lr']))
    if test_acc > best_acc:
        best_acc, best_state = test_acc, deepcopy(model.state_dict())
    print(f'ep {ep:02d}/{CFG["n_epochs"]:02d}  '
          f'loss={sum_loss/n:.4f} (ce={sum_ce/n:.4f} mar={sum_mar/n:.4f} σ̄={sum_sig/n:.4f})  '
          f'test_acc={test_acc:.4f}  margin_med={median_margin:.3f}  '
          f'({time.time()-t0:.0f}s)')

model.load_state_dict(best_state)
print(f'\nBest test accuracy: {best_acc:.4f}')
print(f'Target ≥ 0.92 — {"PASS" if best_acc >= 0.92 else "BELOW TARGET"}')

# Final σ summary
sigma_summary(model, 'after training, post-cap')


ep 01/30  loss=2.7204 (ce=1.3082 mar=1.3894 σ̄=0.4564)  test_acc=0.8601  margin_med=2.404  (35s)
ep 02/30  loss=0.8852 (ce=0.3739 mar=0.4803 σ̄=0.6215)  test_acc=0.9018  margin_med=3.516  (35s)
ep 03/30  loss=0.6003 (ce=0.2399 mar=0.3280 σ̄=0.6488)  test_acc=0.9493  margin_med=4.410  (35s)
ep 04/30  loss=0.4673 (ce=0.1805 mar=0.2537 σ̄=0.6621)  test_acc=0.9523  margin_med=4.786  (35s)
ep 05/30  loss=0.4013 (ce=0.1512 mar=0.2166 σ̄=0.6712)  test_acc=0.9641  margin_med=5.331  (35s)
ep 06/30  loss=0.3546 (ce=0.1305 mar=0.1902 σ̄=0.6792)  test_acc=0.9655  margin_med=5.466  (34s)
ep 07/30  loss=0.3248 (ce=0.1172 mar=0.1734 σ̄=0.6842)  test_acc=0.9702  margin_med=5.472  (34s)
ep 08/30  loss=0.2967 (ce=0.1049 mar=0.1573 σ̄=0.6912)  test_acc=0.9696  margin_med=5.812  (35s)
ep 09/30  loss=0.2687 (ce=0.0938 mar=0.1400 σ̄=0.6977)  test_acc=0.9686  margin_med=5.963  (35s)
ep 10/30  loss=0.2532 (ce=0.0870 mar=0.1309 σ̄=0.7040)  test_acc=0.9700  margin_med=6.257  (35s)
ep 11/30  loss=0.2415 (ce=0.08

In [10]:
# ── Save checkpoint ───────────────────────────────────────────────────────
#
# Two state_dicts are saved:
#   * 'state_dict'                — raw, with parametrize machinery (weight_orig + cap buffer)
#   * 'state_dict_materialized'   — plain post-cap weights (loadable into the
#                                   plain ViTTiny class from notebook 09)
#
# Notebooks 11 / 12 / 13 / 14 / 15 / 16 / 17 / 18 should always load
# `state_dict_materialized`.

# Materialize: build a plain copy and copy POST-CAP weights manually.
# (parametrize.remove_parametrizations is fragile across torch versions
#  when the module re-uses 'weight' — manual copy is simpler and robust.)
model_mat = ViTTiny(
    img_size=CFG['img_size'], patch_size=CFG['patch_size'],
    embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'],
    num_layers=CFG['num_layers'], mlp_ratio=CFG['mlp_ratio'],
    eps_rms=CFG['eps_rms'],
).to(device)

# Load matching biases / pos_embed / RMSNorm γ from the trained model
mat_sd = model_mat.state_dict()
trained_sd = model.state_dict()                     # has weight_orig keys
for k in mat_sd.keys():
    if k in trained_sd:
        mat_sd[k] = trained_sd[k].clone()
    elif k.endswith('.weight'):
        # Pull the post-cap weight from the live `model` (where parametrize
        # exposes the materialized tensor through `.weight`).
        # k example: 'patch_embed.proj.weight'
        mod = model
        for piece in k.split('.')[:-1]:
            mod = getattr(mod, piece)
        with torch.no_grad():
            mat_sd[k] = mod.weight.detach().clone()
model_mat.load_state_dict(mat_sd)

# Sanity: materialized model gives identical outputs to the live SN model
with torch.no_grad():
    _x = torch.randn(4, 1, 28, 28, device=device)
    _a = model(_x); _b = model_mat(_x)
    diff = (_a - _b).abs().max().item()
    print(f'live vs materialized max-abs diff: {diff:.2e}  '
          f'({"OK" if diff < 1e-4 else "MISMATCH — fix before saving"})')

ckpt_path = Path(CFG['run_dir']) / 'model.pt'
torch.save({
    'state_dict':              model.state_dict(),
    'state_dict_materialized': model_mat.state_dict(),
    'cfg':       model.cfg,
    'best_acc':  best_acc,
    'history':   history,
    'training':  {k: CFG[k] for k in ('seed','n_epochs','batch_size','lr','weight_decay',
                                      'margin_kappa','lambda_marg','lambda_sigma','sn_n_iter')},
    'spectral_norm_applied': True,
    'spectral_norm_kind':    'soft_cap_max1',
}, ckpt_path)
print(f'Saved → {ckpt_path}  ({ckpt_path.stat().st_size/1e3:.1f} KB)')
print(f'Best test acc: {best_acc:.4f}')


live vs materialized max-abs diff: 1.24e-05  (OK)
Saved → C:\Users\manya\OneDrive\Desktop\THESIS\formal-verification\runs\vit_tiny_lipmargin/model.pt  (603.5 KB)
Best test acc: 0.9838


In [16]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Define the Google Drive path
drive_path = '/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin'
os.makedirs(drive_path, exist_ok=True)

# Save the checkpoint to Google Drive
ckpt_path_drive = os.path.join(drive_path, 'model.pt')
torch.save({
    'state_dict': model.state_dict(),
    'state_dict_materialized': model_mat.state_dict(),
    'cfg': model.cfg,
    'best_acc': best_acc,
    'history': history,
    'training': {k: CFG[k] for k in ('seed','n_epochs','batch_size','lr','weight_decay',
                                      'margin_kappa','lambda_marg','lambda_sigma','sn_n_iter')},
    'spectral_norm_applied': True,
    'spectral_norm_kind': 'soft_cap_max1',
}, ckpt_path_drive)
print(f'Saved to Google Drive → {ckpt_path_drive}  ({os.path.getsize(ckpt_path_drive)/1e3:.1f} KB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive → /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin/model.pt  (603.5 KB)


In [11]:
pwd

'/content'

In [12]:
cd ../

/


In [13]:
pwd

'/'

In [14]:
ls

bin@                        lib32@                    root/
boot/                       lib64@                    run/
content/                    libx32@                   sbin@
cuda-keyring_1.1-1_all.deb  media/                    srv/
datalab/                    mnt/                      sys/
dev/                        NGC-DL-CONTAINER-LICENSE  tmp/
etc/                        opt/                      tools/
home/                       proc/                     usr/
kaggle/                     python-apt/               var/
lib@                        python-apt.tar.xz*
